In [0]:
client_id     = dbutils.secrets.get(scope="kv-scope", key="sp-client-id")
tenant_id     = dbutils.secrets.get(scope="kv-scope", key="sp-tenant-id")
client_secret = dbutils.secrets.get(scope="kv-scope", key="sp-client-secret")

storage_account = "azurelabadls225"

spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net", client_secret)

RAW_PATH       = f"abfss://raw@{storage_account}.dfs.core.windows.net"
PROCESSED_PATH = f"abfss://processed@{storage_account}.dfs.core.windows.net"
CURATED_PATH   = f"abfss://curated@{storage_account}.dfs.core.windows.net"

print(" Setup complete")

 Setup complete


In [0]:
df_bronze = spark.read.format("delta").load(f"{RAW_PATH}/delta/bronze_yellow_taxi")

print(f"Bronze rows: {df_bronze.count():,}")
df_bronze.printSchema()

Bronze rows: 38,310,226
root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- pipel

In [0]:
from pyspark.sql.functions import col, year, unix_timestamp

df_cleaned = (df_bronze
    # Remove negative or zero fares
    .filter(col("fare_amount") > 0)
    # Remove trips with no distance
    .filter(col("trip_distance") > 0)
    # Remove invalid passenger counts
    .filter(col("passenger_count") > 0)
    .filter(col("passenger_count") <= 6)
    # Remove null locations
    .filter(col("PULocationID").isNotNull())
    .filter(col("DOLocationID").isNotNull())
    # Remove null timestamps
    .filter(col("tpep_pickup_datetime").isNotNull())
    .filter(col("tpep_dropoff_datetime").isNotNull())
    # Remove trips where dropoff is before pickup
    .filter(col("tpep_dropoff_datetime") > col("tpep_pickup_datetime"))
    # Only keep 2023 data — removes wrong year garbage rows
    .filter(year(col("tpep_pickup_datetime")) == 2023)
    # Max trip duration 3 hours — removes impossibly long trips
    .filter(
        ((unix_timestamp("tpep_dropoff_datetime") -
          unix_timestamp("tpep_pickup_datetime")) / 60) <= 180
    )
    # Remove duplicates
    .dropDuplicates(["tpep_pickup_datetime", "tpep_dropoff_datetime",
                     "PULocationID", "DOLocationID", "fare_amount"])
)

print(f"Rows after cleaning: {df_cleaned.count():,}")
print(f"Rows removed: {df_bronze.count() - df_cleaned.count():,}")

Rows after cleaning: 35,574,985
Rows removed: 2,735,241


In [0]:
from pyspark.sql.functions import broadcast

# Read zone lookup — small table (265 rows) perfect for broadcast join
df_zones = (spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_PATH}/lookup/taxi_zone_lookup.csv")
)

print(f"Zone lookup rows: {df_zones.count()}")
df_zones.show(3)

# Broadcast join — send the small zones table to ALL executors
# This avoids a shuffle of the 38M row table
df_silver = (df_cleaned
    .join(broadcast(df_zones.alias("pickup_zone")),
          df_cleaned.PULocationID == col("pickup_zone.LocationID"),
          "left")
    .withColumnRenamed("Borough", "pickup_borough")
    .withColumnRenamed("Zone", "pickup_zone_name")
    .withColumnRenamed("service_zone", "pickup_service_zone")
    .drop("LocationID")
    .join(broadcast(df_zones.alias("dropoff_zone")),
          df_cleaned.DOLocationID == col("dropoff_zone.LocationID"),
          "left")
    .withColumnRenamed("Borough", "dropoff_borough")
    .withColumnRenamed("Zone", "dropoff_zone_name")
    .withColumnRenamed("service_zone", "dropoff_service_zone")
    .drop("LocationID")
)

print(f"✅ Broadcast join complete")
print(f"Silver columns: {len(df_silver.columns)}")

Zone lookup rows: 265
+----------+-------+--------------------+------------+
|LocationID|Borough|                Zone|service_zone|
+----------+-------+--------------------+------------+
|         1|    EWR|      Newark Airport|         EWR|
|         2| Queens|         Jamaica Bay|   Boro Zone|
|         3|  Bronx|Allerton/Pelham G...|   Boro Zone|
+----------+-------+--------------------+------------+
only showing top 3 rows

✅ Broadcast join complete
Silver columns: 28


In [0]:
from pyspark.sql.functions import (unix_timestamp, round, hour, 
                                    dayofweek, month, year, lit)

df_silver_enriched = (df_silver
    # Trip duration in minutes
    .withColumn("trip_duration_mins",
        round((unix_timestamp("tpep_dropoff_datetime") - 
               unix_timestamp("tpep_pickup_datetime")) / 60, 2))
    # Time features
    .withColumn("pickup_hour",    hour("tpep_pickup_datetime"))
    .withColumn("pickup_day",     dayofweek("tpep_pickup_datetime"))
    .withColumn("pickup_month",   month("tpep_pickup_datetime"))
    .withColumn("pickup_year",    year("tpep_pickup_datetime"))
    # Silver metadata
    .withColumn("silver_pipeline", lit("nyc_taxi_silver"))
)

print(f"✅ Derived columns added")
print(f"Final silver columns: {len(df_silver_enriched.columns)}")

✅ Derived columns added
Final silver columns: 34


In [0]:
silver_path = f"{PROCESSED_PATH}/delta/silver_yellow_taxi"

(df_silver_enriched
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("pickup_year", "pickup_month")
    .save(silver_path)
)

print("✅ Silver layer written!")

✅ Silver layer written!


In [0]:
df_verify = spark.read.format("delta").load(f"{PROCESSED_PATH}/delta/silver_yellow_taxi")

print(f"Total rows:    {df_verify.count():,}")
print(f"Total columns: {len(df_verify.columns)}")
df_verify.show(3)

Total rows:    35,574,985
Total columns: 34
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+--------------------+--------------------+---------------+--------------+----------------+-------------------+---------------+-----------------+--------------------+------------------+-----------+----------+------------+-----------+---------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee| ingestion_timestamp|         source_file|  pipeline_name|pickup_borough|pickup_zone_name|pickup_service_zone|dropoff_borough|dropoff_zone_name|dropoff_service_zone|tr

In [0]:
print("=" * 60)
print("🥈 SILVER LAYER RESULTS")
print("=" * 60)

df_silver = spark.read.format("delta").load(f"{PROCESSED_PATH}/delta/silver_yellow_taxi")

print(f"\nTotal rows: {df_silver.count():,}")
print(f"Total columns: {len(df_silver.columns)}")

print("\n📊 ROW COUNT PER MONTH:")
df_silver.groupBy("pickup_month") \
    .agg(count("*").alias("trip_count")) \
    .orderBy("pickup_month") \
    .show()

print("\n📊 BOROUGH DISTRIBUTION:")
df_silver.groupBy("pickup_borough") \
    .agg(count("*").alias("trip_count")) \
    .orderBy(col("trip_count").desc()) \
    .show()

print("\n👀 SAMPLE ROWS:")
df_silver.show(5, truncate=False)

🥈 SILVER LAYER RESULTS

Total rows: 35,574,985
Total columns: 34

📊 ROW COUNT PER MONTH:


---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-8116629752475033>, line 12
      8 print(f"Total columns: {len(df_silver.columns)}")
     10 print("\n📊 ROW COUNT PER MONTH:")
     11 df_silver.groupBy("pickup_month") \
---> 12     .agg(count("*").alias("trip_count")) \
     13     .orderBy("pickup_month") \
     14     .show()
     16 print("\n📊 BOROUGH DISTRIBUTION:")
     17 df_silver.groupBy("pickup_borough") \
     18     .agg(count("*").alias("trip_count")) \
     19     .orderBy(col("trip_count").desc()) \
     20     .show()

NameError: name 'count' is not defined